# 02 — Bronze Layer Validation and Profiling
## Customer Analytics Platform

Purpose: Validate Bronze Delta tables, check data quality,
and understand the dataset before Silver transformation.

In [0]:
# set bronze path and load all delta tables for validation

bronze_path = "/Volumes/workspace/default/olist_raw_data/bronze"

customers_df   = spark.read.format("delta").load(f"{bronze_path}/customers")
orders_df      = spark.read.format("delta").load(f"{bronze_path}/orders")
order_items_df = spark.read.format("delta").load(f"{bronze_path}/order_items")
payments_df    = spark.read.format("delta").load(f"{bronze_path}/payments")
reviews_df     = spark.read.format("delta").load(f"{bronze_path}/reviews")
products_df    = spark.read.format("delta").load(f"{bronze_path}/products")
sellers_df     = spark.read.format("delta").load(f"{bronze_path}/sellers")
geolocation_df = spark.read.format("delta").load(f"{bronze_path}/geolocation")
translation_df = spark.read.format("delta").load(f"{bronze_path}/translation")

print("All Bronze Delta tables loaded successfully")

In [0]:
# check null counts across key tables to understand data quality

from pyspark.sql.functions import col, sum as spark_sum

def null_check(df, table_name):
    print(f"\n{table_name} null counts:")
    print("-" * 40)
    null_counts = df.select([
        spark_sum(col(c).isNull().cast("int")).alias(c) 
        for c in df.columns
    ]).collect()[0].asDict()
    
    for column, count in null_counts.items():
        if count > 0:
            print(f"  {column}: {count} nulls")
    
null_check(customers_df,   "customers")
null_check(orders_df,      "orders")
null_check(order_items_df, "order_items")
null_check(payments_df,    "payments")
null_check(reviews_df,     "reviews")
null_check(products_df,    "products")
null_check(sellers_df,     "sellers")

print("\nNull check complete")

In [0]:
# check schema of the most important tables for silver planning

print("orders schema:")
print("-" * 40)
orders_df.printSchema()

print("order_items schema:")
print("-" * 40)
order_items_df.printSchema()

print("reviews schema:")
print("-" * 40)
reviews_df.printSchema()

In [0]:
# check order status distribution to understand data better

print("Order status breakdown:")
print("-" * 40)
orders_df.groupBy("order_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

In [0]:
# check date range of orders to understand time period of data

from pyspark.sql.functions import min, max

orders_df.select(
    min("order_purchase_timestamp").alias("first_order"),
    max("order_purchase_timestamp").alias("last_order")
).show()

In [0]:
# check payment value stats to understand revenue distribution

from pyspark.sql.functions import round as spark_round

payments_df.select(
    spark_round(payments_df.payment_value.cast("double"), 2)
    .alias("payment_value")
).summary(
    "count", "mean", "min", "25%", "75%", "max"
).show()

In [0]:
# check top 10 product categories by order volume

from pyspark.sql.functions import col

# join order_items with products and translation
category_analysis = order_items_df \
    .join(products_df, "product_id", "left") \
    .join(translation_df, "product_category_name", "left") \
    .groupBy("product_category_name_english") \
    .count() \
    .orderBy("count", ascending=False) \
    .limit(10)

category_analysis.show(truncate=False)

## Bronze Layer Summary

Data Quality Findings:
- customers and order_items have zero nulls
- orders has nulls in delivery date columns 
  (expected — undelivered orders)
- reviews has nulls in comment columns 
  (customers who left no text feedback)

Key Business Insights:
- 96.5% of orders successfully delivered
- Data spans September 2016 to October 2018
- Average order value is R$ 154 BRL
- Top category is bed, bath and table products
- High value threshold likely above R$ 172 BRL

Action for Silver Layer:
- Filter only delivered orders for CLV model
- Handle nulls in delivery dates
- Clean review text for RAG layer
- Fix data types where needed